# Trade Validation and Exception Tracker (Simulated Data)

In [29]:
import pandas as pd
xl = pd.read_excel("simulated_trades.xlsx", sheet_name=None)
trades = xl["trades"]
cps = xl["counterparties"]
prods = xl["products"]
trades.head()

,trade_id,trade_date,settlement_date,counterparty_id,product_id,side,quantity,price,currency,booking_desk,entry_method
0,T00001,2026-09-09,2026-09-11,CP025,EQ0008,SELL,15900.0,93.62,USD,Cash Equities,Electronic
1,T00002,2026-09-09,2026-09-11,CP022,GB0001,BUY,12000.0,101.80,EUR,ETF Desk,Electronic
2,T00003,2026-09-09,2026-09-11,CP016,EQ0003,SELL,4900.0,206.45,EUR,London Credit,Electronic
3,T00004,2026-09-09,2026-09-11,CP031,EQ0009,BUY,23800.0,24.30,EUR,ETF Desk,Electronic
4,T00005,2026-09-09,2026-09-11,CP028,GB0004,BUY,16000.0,106.16,GBP,ETF Desk,Electronic


In [30]:
trades.shape

(2000, 11)

In [31]:
df = trades.copy()
df["missing_field"] = df[["counterparty_id", "quantity", "price", "settlement_date"]].isna().any(axis=1)
df["missing_field"].sum()

np.int64(25)

In [32]:
df = df.merge(prods, on="product_id", how="left", suffixes=("", "_product"))
df = df.merge(cps, on="counterparty_id", how="left")
df.shape

(2000, 19)

In [33]:
df["unknown_counterparty"] = df["counterparty_id"].notna() & df["status"].isna()
df["unknown_product"] = df["asset_class"].isna()
print(df["unknown_counterparty"].sum(), df["unknown_product"].sum())

25 20


In [34]:
df["settle_before_trade"] = pd.to_datetime(df["settlement_date"]) < pd.to_datetime(df["trade_date"])
df["settle_on_weekend"] = pd.to_datetime(df["settlement_date"]).dt.weekday >= 5
print(df["settle_before_trade"].sum(), df["settle_on_weekend"].sum())

20 22


In [35]:
df["quantity_over_limit"] = df["quantity"] > df["max_quantity"]
df["suspended_counterparty"] = df["status"] == "Suspended"
df["currency_mismatch"] = df["asset_class"].notna() & (df["currency"] != df["currency_product"])
print(df["quantity_over_limit"].sum(), df["suspended_counterparty"].sum(), df["currency_mismatch"].sum())

25 15 15


In [36]:
df["duplicate_trade_id"] = df["trade_id"].duplicated(keep=False)
df["duplicate_trade_id"].sum()

np.int64(30)

In [37]:
rules = ["missing_field", "unknown_counterparty", "unknown_product", "settle_before_trade",
         "settle_on_weekend", "quantity_over_limit", "suspended_counterparty",
         "currency_mismatch", "duplicate_trade_id"]

df["exception_count"] = df[rules].sum(axis=1)
df[df["exception_count"] > 0]["trade_id"].nunique()

180

In [38]:
exceptions = df[df["exception_count"] > 0].copy()
exceptions["issues"] = exceptions[rules].apply(lambda row: ", ".join([c for c in rules if row[c]]), axis=1)
print(len(exceptions))
exceptions[["trade_id", "trade_date", "issues"]].head()

195


,trade_id,trade_date,issues
18,T00019,2026-09-09,duplicate_trade_id
102,T00103,2026-09-09,missing_field
103,T00104,2026-09-09,unknown_counterparty
104,T00105,2026-09-09,unknown_counterparty
105,T00106,2026-09-09,unknown_counterparty


In [39]:
as_of = pd.Timestamp("2026-09-30")
exceptions["days_open"] = (as_of - pd.to_datetime(exceptions["trade_date"])).dt.days
exceptions["days_open"].describe()

,days_open
count,195.000000
mean,12.066667
std,6.242576
min,0.000000
25%,7.000000
50%,13.000000
75%,19.000000
max,21.000000


In [40]:
bins = [-1, 2, 5, 10, 100]
labels = ["0-2 days", "3-5 days", "6-10 days", "Over 10 days"]
exceptions["age_bucket"] = pd.cut(exceptions["days_open"], bins=bins, labels=labels)
exceptions["age_bucket"].value_counts().sort_index()

,count
age_bucket,
0-2 days,18
3-5 days,14
6-10 days,52
Over 10 days,111


In [41]:
escalate_to = {
    "0-2 days": "Operations analyst",
    "3-5 days": "Team lead",
    "6-10 days": "Operations manager",
    "Over 10 days": "Head of operations",
}
exceptions["escalate_to"] = exceptions["age_bucket"].map(escalate_to)
exceptions[["trade_id", "days_open", "age_bucket", "escalate_to"]].head()

,trade_id,days_open,age_bucket,escalate_to
18,T00019,21,Over 10 days,Head of operations
102,T00103,21,Over 10 days,Head of operations
103,T00104,21,Over 10 days,Head of operations
104,T00105,21,Over 10 days,Head of operations
105,T00106,21,Over 10 days,Head of operations


In [42]:
by_rule = exceptions[rules].sum().sort_values(ascending=False)
by_rule

,0
duplicate_trade_id,30
missing_field,25
unknown_counterparty,25
quantity_over_limit,25
settle_on_weekend,22
settle_before_trade,20
unknown_product,20
suspended_counterparty,15
currency_mismatch,15


In [43]:
df["has_exception"] = df["exception_count"] > 0
(df.groupby("entry_method")["has_exception"].mean() * 100).round(1)

,has_exception
entry_method,
Electronic,4.2
Manual,35.8


In [44]:
summary_rules = by_rule.reset_index()
summary_rules.columns = ["rule", "exceptions"]

summary_age = exceptions["age_bucket"].value_counts().sort_index().reset_index()
summary_age.columns = ["age_bucket", "exceptions"]

summary_method = df.groupby("entry_method")["has_exception"].agg(["count", "sum"]).reset_index()
summary_method.columns = ["entry_method", "total_trades", "trades_with_exception"]

export_cols = ["trade_id", "trade_date", "settlement_date", "counterparty_id", "product_id",
               "side", "quantity", "price", "currency", "booking_desk", "entry_method",
               "issues", "days_open", "age_bucket", "escalate_to"]

with pd.ExcelWriter("exception_report.xlsx") as writer:
    exceptions[export_cols].to_excel(writer, sheet_name="exceptions", index=False)
    summary_rules.to_excel(writer, sheet_name="by_rule", index=False)
    summary_age.to_excel(writer, sheet_name="by_age", index=False)
    summary_method.to_excel(writer, sheet_name="by_method", index=False)

from google.colab import files
files.download("exception_report.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [45]:
rule_flags = exceptions[rules].astype(int)
dash = exceptions[export_cols].join(rule_flags)

by_method = df.groupby("entry_method")["has_exception"].agg(["count", "sum"]).reset_index()
by_method.columns = ["entry_method", "total_trades", "trades_with_exception"]
by_method["exception_rate"] = by_method["trades_with_exception"] / by_method["total_trades"]

with pd.ExcelWriter("exception_data_for_dashboard.xlsx") as writer:
    dash.to_excel(writer, sheet_name="exceptions", index=False)
    by_method.to_excel(writer, sheet_name="by_method", index=False)

from google.colab import files
files.download("exception_data_for_dashboard.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [46]:
import sqlite3
con = sqlite3.connect(":memory:")
trades.to_sql("trades", con, index=False)
cps.to_sql("counterparties", con, index=False)
prods.to_sql("products", con, index=False)
def q(sql):
 return pd.read_sql(sql, con)

In [47]:
q("""SELECT COUNT(*) AS exceptions FROM trades
 WHERE counterparty_id IS NULL OR quantity IS NULL
 OR price IS NULL OR settlement_date IS NULL""")

,exceptions
0,25


In [48]:
q("""SELECT COUNT(*) AS exceptions FROM trades t
 LEFT JOIN counterparties c ON t.counterparty_id = c.counterparty_id
 WHERE t.counterparty_id IS NOT NULL AND c.counterparty_id IS NULL""")


,exceptions
0,25


In [49]:
q("""SELECT COUNT(*) AS exceptions FROM trades t
 JOIN products p ON t.product_id = p.product_id
 WHERE t.quantity > p.max_quantity""")

,exceptions
0,25


In [50]:
q("""SELECT COUNT(*) AS exceptions FROM trades
 WHERE strftime('%w', settlement_date) IN ('0','6')""")

,exceptions
0,22


In [51]:
q("""SELECT COUNT(*) AS duplicated_ids FROM
 (SELECT trade_id FROM trades GROUP BY trade_id HAVING COUNT(*) > 1)""")

,duplicated_ids
0,15
